# Lab 6.3 &mdash; Adequacy, Re-querying and Multi-Hop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Judge your own retrieval &mdash; does it actually contain what was asked for?
- Build the retrieve &rarr; judge &rarr; re-query loop as a compiled <code>StateGraph</code>
- Re-query with a term the first hop taught you
- Stop: a hop budget, a repeat detector, and &lsquo;I could not find it&rsquo; as a real outcome

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a chunked
> `Document`, a Chroma collection, a bound tool, a compiled graph, a parser), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code in
> front of the sandbox model; that is the part worth watching. The score line is feedback, not a
> grade.

> **This is what makes it agentic.** A pipeline retrieves once. Everything in this lab
> is the loop a pipeline cannot have, and the stops that keep it from running away.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents about the same payments. Read 3.2: the rule and the exception
# that qualifies it are adjacent sentences, which is the whole of Lab 6.1's first lesson. Note
# also what is NOT here -- nothing mentions FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- the embedding model (nothing to fill in)
# The sandbox has no egress, and chromadb's DEFAULT embedding function downloads about 80 MB
# of ONNX model the first time it is called. So this module brings its own: one hashed bucket
# per meaningful word, normalised to unit length. It is arithmetic rather than learning, which
# is the point -- it runs offline, it is deterministic, and you can read every line of it.
#
# What it CAN do: score two texts by the words they share. What it CANNOT do: match meaning
# with no words in common. Lab 6.2 is about living with exactly that.
import re, math, hashlib
from langchain_core.embeddings import Embeddings

STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def content_words(text: str) -> list:
    """The words worth indexing: lower-cased, no punctuation, no stop words."""
    return [w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1]


class LabEmbeddings(Embeddings):
    """A tiny embedding model you can read. Same interface as any other LangChain embedding."""

    dim = 1024                      # enough buckets that two different words rarely collide

    def _vector(self, text: str) -> list:
        vec = [0.0] * self.dim
        for word in content_words(text):
            bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % self.dim
            vec[bucket] += 1.0
        length = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [x / length for x in vec]      # unit length, so cosine is just a dot product

    def embed_documents(self, texts: list) -> list:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list:
        return self._vector(text)


print("embeddings:", LabEmbeddings.dim, "dimensions, offline, deterministic")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
# Exactly what you built in Lab 6.1: split on headings, index in Chroma, search with a floor.
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma

FLOOR = 0.20            # the similarity a chunk must clear to be used at all (Lab 6.1)

def section_chunks() -> list:
    """One Document per '##' section, with the heading kept in the text and in the metadata."""
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "section")],
                                          strip_headers=False)
    out = []
    for name, text in DOCS.items():
        for chunk in splitter.split_text(text):
            chunk.metadata["source"] = name
            out.append(chunk)
    return out


_store = None
def store():
    """The Chroma collection, built once, on first use."""
    global _store
    if _store is None:
        chunks = section_chunks()
        _store = Chroma(collection_name="module6-corpus",
                        embedding_function=LabEmbeddings(),
                        persist_directory=os.path.join(WORK, "chroma"),
                        collection_configuration={"hnsw": {"space": "cosine"}})
        # ids derived from the chunk, so re-running this notebook updates instead of duplicating
        _store.add_documents(chunks, ids=[f"{c.metadata['source']}#{c.metadata['section']}"
                                          for c in chunks])
    return _store


def search(query: str, k: int = 4, floor: float = 0.0, where: dict | None = None) -> list:
    """Top-k from the store as plain dicts, with anything below `floor` dropped."""
    hits = store().similarity_search_with_score(query, k=k, filter=where)
    out = []
    for doc, distance in hits:
        similarity = 1.0 - distance         # cosine space: 1.0 identical, 0.0 nothing in common
        if similarity >= floor:
            out.append({"score": round(similarity, 3), "text": doc.page_content,
                        "source": doc.metadata["source"], "section": doc.metadata["section"]})
    return out


print(f"index ready: {len(store().get()['ids'])} chunks")

## Concept

The first retrieval usually returns something. The question is whether it returns *enough*, and
on this corpus that is answerable without a model: **did what came back contain the things the
question asked about?**

When it did not, the results still tell you something &mdash; they hand you the corpus's own
vocabulary, which is exactly what the second query needed. That loop is a graph:

```
retrieve -> judge -> (adequate?) -> report
                  \-> re-query -> retrieve -> ...
```

You built graphs in Module 3. Same `StateGraph`, same conditional edge, same reason for a budget:
a cycle without one is a bill.

## Section 1 &mdash; Was that enough?

An adequacy test that is honest has to be able to say no. `coverage` measures how much of what
the question needed actually appeared; you decide how much is enough.

In [ ]:
def coverage(need_terms: list, results: list) -> float:
    """How much of what the question needed actually appeared in what came back, 0.0 to 1.0."""
    need = {t.lower() for t in need_terms}
    if not need:
        return 0.0
    covered = set(content_words(" ".join(r["text"] for r in results)))
    return len(need & covered) / len(need)


def missing(need_terms: list, results: list) -> list:
    """What the question asked about that the results never mention."""
    covered = set(content_words(" ".join(r["text"] for r in results)))
    return sorted(t for t in need_terms if t.lower() not in covered)


def adequate(need_terms: list, results: list) -> bool:
    """Is this retrieval enough to answer from?"""
    if not results:
        return False
    return coverage(need_terms, results) >= 1.0     # every term, or it is not an answer yet

In [ ]:
# --- Self-check: Section 1   (over the real index -- no model)
SANCTIONS = ("what does a sanctions review need", ["sanctions", "compliance"])
HEDGING   = ("what is the JPY hedging policy",    ["hedging"])
HALF      = ("what does a sanctions review need", ["sanctions", "hedging"])   # one of two present

check("an empty retrieval is never adequate",
      lambda: adequate(["anything"], []) is False)
check("a retrieval that covers everything asked for IS adequate",
      lambda: adequate(SANCTIONS[1], search(SANCTIONS[0], k=3)) is True)
check("one that covers nothing is NOT, even though it returned rows",
      lambda: adequate(HEDGING[1], search(HEDGING[0], k=3)) is False,
      "three chunks came back and none is about hedging -- a length check would pass this")
check("half covered is not enough either",
      lambda: adequate(HALF[1], search(HALF[0], k=3)) is False,
      "that is the bar: a partial retrieval is a wrong answer waiting to be written")
check("coverage is a number you can log, not just a verdict",
      lambda: abs(coverage(HALF[1], search(HALF[0], k=3)) - 0.5) < 1e-9)
check("and it names what was missing",
      lambda: missing(HEDGING[1], search(HEDGING[0], k=3)) == ["hedging"],
      "'the corpus has nothing on hedging' is a useful answer; 'I don't know' is not")

## Section 2 &mdash; The re-query, and the reason it is not a rewrite

Hop two's query contains a word you could not have known before hop one ran. That is what
&ldquo;multi-hop&rdquo; means &mdash; not three searches, but three searches where each is written from the
last one's answer.

On this corpus the handle is the reason code. `follow_up` reads the retrieved text and returns
the first code it did not already ask about.

In [ ]:
CODE_RE = re.compile(r"\b(R\d{2}|[A-Z]{2,}_[A-Z_]+)\b")

def follow_up(results: list, asked: str):
    """A query built from a term the results just taught you, or None if they taught nothing."""
    found = []
    for r in results:
        found += CODE_RE.findall(r["text"])
    fresh = [f for f in found if f.lower() not in (asked or "").lower()]
    return fresh[0] if fresh else None

In [ ]:
# --- Self-check: Section 2   (string handling over real retrievals -- no model)
IBAN_Q = "what happens with a wrong beneficiary iban"

check("the first hop on a beneficiary question finds section 3.3",
      lambda: search(IBAN_Q, k=2)[0]["section"].startswith("3.3"))
check("and the results teach it a code the question never contained",
      lambda: follow_up(search(IBAN_Q, k=2), IBAN_Q) in ("INVALID_IBAN", "R04"),
      "that term came out of the corpus -- no rewrite of the question could have produced it")
check("a term already in the query is not chased again",
      lambda: follow_up(search("INVALID_IBAN", k=2), "INVALID_IBAN") != "INVALID_IBAN")
check("results that name no codes teach nothing, and say so",
      lambda: follow_up([{"text": "no codes here at all"}], "x") is None,
      "None, not an empty string -- the loop below branches on it")

## Section 3 &mdash; The loop, as a graph

Four nodes. `retrieve` searches, `judge` scores the result, `requery` swaps in the new query, and
`report` writes the outcome. The conditional edge after `judge` is where every stop lives:
adequate, nothing left to chase, budget spent, or asking the same thing twice.

`hops` uses the `Annotated[list, add]` reducer from Module 3, so each pass appends its trace
instead of replacing it.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

MAX_HOPS = 3

class HuntState(TypedDict):
    question: str
    need: list
    query: str                          # the query THIS hop will send
    hops: Annotated[list, add]          # one entry per hop, appended
    results: list
    adequate: bool
    next_query: str | None
    outcome: str


def retrieve(state: HuntState) -> dict:
    hits = search(state["query"], k=3)
    return {"results": hits,
            "hops": [{"query": state["query"], "sections": [h["section"] for h in hits]}]}


def judge(state: HuntState) -> dict:
    ok = adequate(state["need"], state["results"])
    return {"adequate": ok,
            "next_query": None if ok else follow_up(state["results"], state["query"])}


def requery(state: HuntState) -> dict:
    return {"query": state["next_query"]}

In [ ]:
def report(state: HuntState) -> dict:
    """The last node. Say what happened, and hand back only what you may answer from."""
    if state["adequate"]:
        return {"outcome": "answered"}
    return {"outcome": "not found", "results": []}   # inadequate evidence is not evidence


def next_step(state: HuntState) -> str:
    """The conditional edge: go round again, or stop? Returns the KEY of the next branch."""
    if state["adequate"]:
        return "report"
    if state["next_query"] is None:                                   # nothing left to chase
        return "report"
    if len(state["hops"]) >= MAX_HOPS:                                # budget spent
        return "report"
    if any(h["query"] == state["next_query"] for h in state["hops"]): # asked that already
        return "report"
    return "requery"

In [ ]:
def build_hunt():
    """Wire the four nodes into a graph with one cycle, and compile it."""
    g = StateGraph(HuntState)
    g.add_node("retrieve", retrieve)
    g.add_node("judge", judge)
    g.add_node("requery", requery)
    g.add_node("report", report)

    g.add_edge(START, "retrieve")
    g.add_edge("retrieve", "judge")
    g.add_conditional_edges("judge", next_step, {"requery": "requery", "report": "report"})
    g.add_edge("requery", "retrieve")        # the backward edge -- this is the cycle
    g.add_edge("report", END)
    return g.compile()


def hunt(question: str, need: list) -> dict:
    """Run the loop for one question."""
    return build_hunt().invoke({"question": question, "need": need, "query": question,
                                "hops": [], "results": [], "adequate": False,
                                "next_query": None, "outcome": ""})

In [ ]:
# --- Self-check: Section 3   (a REAL compiled graph, running the real index -- still no model)
check("the graph compiles",
      lambda: build_hunt() is not None)
check("an answerable question is answered on the first hop",
      lambda: hunt(*SANCTIONS)["outcome"] == "answered" and len(hunt(*SANCTIONS)["hops"]) == 1)
check("and it hands back the evidence it answered from",
      lambda: len(hunt(*SANCTIONS)["results"]) == 3)
check("an unanswerable question ends as 'not found', not as a wrong answer",
      lambda: hunt(*HEDGING)["outcome"] == "not found")
check("and it hands back NOTHING to answer from",
      lambda: hunt(*HEDGING)["results"] == [],
      "handing the irrelevant chunks back anyway is how a refusal becomes a hallucination")
check("it went round again before giving up",
      lambda: len(hunt(*HEDGING)["hops"]) >= 2,
      "hop two used a term the corpus supplied -- it failed honestly, not lazily")
check("the hop budget is never exceeded",
      lambda: len(hunt(*HEDGING)["hops"]) <= MAX_HOPS)
check("the cycle cannot ask the same thing twice",
      lambda: len({h["query"] for h in hunt(*HEDGING)["hops"]})
              == len(hunt(*HEDGING)["hops"]))
check("the trace records every query it sent",
      lambda: all(set(h) == {"query", "sections"} for h in hunt(*HEDGING)["hops"]),
      "this is the log line you will want when someone asks why it said no")

def _traces():
    for question, need in (SANCTIONS, HEDGING, (IBAN_Q, ["r04", "originator"])):
        out = hunt(question, need)
        print(f"  {out['outcome']:10} {question[:44]}")
        for i, h in enumerate(out["hops"], 1):
            print(f"      hop {i}: {h['query'][:38]:40} -> {h['sections']}")
        if out["outcome"] == "not found":
            print(f"      missing: {missing(need, search(question, k=3))}")
guard(_traces)

## Run it for real &mdash; let the model be the judge

Same two questions, but `adequate` is now the model. The one to watch is the second: a model asked
&ldquo;is this enough?&rdquo; about three irrelevant chunks has every incentive to say yes.

In [ ]:
if llm_ready():
    def _model_judge():
        for question, need in (SANCTIONS, HEDGING):
            results = search(question, k=3)
            context = "\n".join(f"- [{r['section']}] {r['text'][:150]}" for r in results)
            verdict = ask(f"Question: {question}\n\nRetrieved:\n{context}\n\n"
                          "Can this question be answered from the retrieved text alone? "
                          "Reply YES or NO, then one short sentence.",
                          system="Begin your reply with YES or NO.")
            print(f"  {question}")
            print(f"      model     : {verdict.strip()[:140]}")
            print(f"      coverage  : {coverage(need, results):.0%} -> "
                  f"{'adequate' if adequate(need, results) else 'not adequate'}")
            print()
    guard(_model_judge)

### Read it

If the model says YES to the hedging question, you have watched the failure this lab exists to
prevent: the retrieval was inadequate, the judge was the same kind of thing that will write the
answer, and nothing stopped it.

The coverage check is crude and cannot be talked round. It also cannot tell a paraphrase from a
gap, which is why it is a floor and not a ceiling &mdash; in production you want both, the cheap
mechanical check underneath and the model for the judgements it is too blunt to make.

Notice what the graph bought you beyond the loop itself: every stop is one line in `next_step`,
and every run leaves a trace of exactly which queries were sent. Both of those are the difference
between an agent you can operate and one you can only demo.

In [ ]:
score()

## Your turn

1. Lower the adequacy bar to 0.5 and re-run `hunt(*HALF)`. It now answers. Read the evidence it
   answered from and decide whether you would sign that answer.
2. `follow_up` chases reason codes because that is what this corpus is made of. What is the
   equivalent handle in your corpus &mdash; a ticket id, a product code, a section number? Write the
   regex and see how far a chain gets.
3. Add a wall-clock deadline to `next_step` as well as the hop budget, then make one search slow.
   Which stop fires first, and which one would you actually have wanted?